In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

df = pd.read_csv("titanic_prepared.csv", index_col=0)
Y = df['label']
X = df.drop('label', axis= 1)

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size= 0.1)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size= 0.15, random_state= 42)

In [9]:
XGBoost_parameters = {
    'n_estimators': [100, 200],        
    'max_depth': [3, 5, 7, 9],         
    'learning_rate': [0.05, 0.1],      
}

XGBoost_Grid = GridSearchCV(XGBClassifier(random_state= 42), XGBoost_parameters, cv= 5, scoring= 'accuracy')
XGBoost_Grid.fit(X_train, Y_train)
XGBoost_best = XGBoost_Grid.best_estimator_

In [10]:
#Logistic Regression
logr_parameters = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'solver': ['liblinear', 'lbfgs']
}

logr_Grid = GridSearchCV(LogisticRegression(max_iter= 10000,random_state= 42), logr_parameters, cv= 5, scoring= 'accuracy')
logr_Grid.fit(X_train, Y_train)
logr_best = logr_Grid.best_estimator_

In [11]:
dTree_parameters = {
'max_depth' : np.arange(2, 21),
'criterion':['gini','entropy']
}
dTree_Grid = GridSearchCV(DecisionTreeClassifier(random_state= 42), dTree_parameters, cv = 5, scoring= 'accuracy')
dTree_Grid.fit(X_train, Y_train)
dTree_best = dTree_Grid.best_estimator_

In [ ]:
models = {
    'XGBoost': XGBoost_best,
    'LogRegression': logr_best,
    'DecisionTree': dTree_best
}

results = {}

for name, model in models.items():
    # дообучаем финальную модель на train
    model.fit(X_train, Y_train)
    
    # предсказываем на тесте
    y_pred = model.predict(X_test)
    
    #вычисляем долю правильных ответов
    accuracy = accuracy_score(Y_test, y_pred)
    
    results[name] = accuracy

results


{'XGBoost': 0.8900602409638554,
 'LogRegression': 0.8674698795180723,
 'DecisionTree': 0.8900602409638554}

In [ ]:
# Обучаем лучшее дерево решений на всех признаках
dTree_best.fit(X_train, Y_train)

# Извлекаем важности признаков после обучения
importances = dTree_best.feature_importances_

# Получаем индексы признаков, отсортированные по важности
indices = np.argsort(importances)

features = X_train.columns

features_num = [2]

results = {}

# Перебираем количество признаков 
for num in features_num:

    # Берём num последних индексов — это самые важные признаки
    ind = indices[-num:]

    feat = features[ind]

    # Создаём таблицы train и test только с нужными признаками
    X_train_crop = X_train[feat]
    X_test_crop = X_test[feat]

    # Обучаем дерево заново, но уже только на выбранных признаках
    dTree_best.fit(X_train_crop, Y_train)

    # Предсказываем на тестовой выборке, оставив только эти фичи
    y_pred = dTree_best.predict(X_test_crop)

    # Считаем точность модели на двух самых важных признаках
    accuracy = accuracy_score(Y_test, y_pred)

    # Сохраняем accuracy для данного количества признаков
    results[str(num)] = accuracy

# Показываем итоговые точности
results


{'2': 0.8674698795180723}